# Demand Forecasting

In this notebook, we forecast weekly SKU-level demand.
Forecasting is done in a segment-aware manner, based on the ABC/XYZ
classification created earlier.

The goal is not just low error, but forecasts that are stable and usable
for inventory planning.

In [15]:
import pandas as pd
import numpy as np
import sys
sys.path.append('../')

from sklearn.metrics import mean_absolute_percentage_error, root_mean_squared_error
from sklearn.linear_model import LinearRegression

from src.forecasting import (
    train_test_split_time,
    naive_forecast,
    prepare_lag_features,
    evaluate_forecast
)

pd.set_option("display.max_columns", None)

DATA_PATH_PROCESSED = "../data/processed"

In [16]:
try:
    df = pd.read_csv(f"{DATA_PATH_PROCESSED}/feature_engineered_with_segments.csv")
    print("Segmented data loaded successfully")
except FileNotFoundError:
    print(f"File not found at {DATA_PATH_PROCESSED}/feature_engineered_with_segments.csv. Please ensure the file exists and the path is correct.")

df["date"] = pd.to_datetime(df["date"])

Segmented data loaded successfully


## Weekly Aggregation

Inventory replenishment is planned weekly, so we aggregate demand
to weekly SKU-level data.

In [17]:
weekly_df = (
    df
    .set_index("date")
    .groupby(["sku_id", "SKU_segment"])
    .resample("W")["units_sold"]
    .sum()
    .reset_index()
)

weekly_df.head()

,sku_id,SKU_segment,date,units_sold
0,SKU0001,AX,2024-01-07,125
1,SKU0001,AX,2024-01-14,235
2,SKU0001,AX,2024-01-21,179
3,SKU0001,AX,2024-01-28,221
4,SKU0001,AX,2024-02-04,204


## Train-Test Split

We use a time-based split to avoid data leakage.
The last 20% of weeks are used for testing.

In [18]:
def train_test_split_time(series, test_size=0.2):
    split_idx = int(len(series) * (1 - test_size))
    return series.iloc[:split_idx], series.iloc[split_idx:]

## Baseline Model: Naive Forecast

The naive model uses last week’s demand as the forecast.
Any advanced model must beat this baseline.

In [19]:
def naive_forecast(train, test):
    return np.repeat(train.iloc[-1], len(test))

## Lag-Based Regression Model

A simple regression model using lagged demand as a predictor.
This works well for stable and moderately seasonal SKUs.

In [20]:
def prepare_lag_features(series, lags=[1, 2, 4]):
    data = pd.DataFrame({"y": series})
    for lag in lags:
        data[f"lag_{lag}"] = series.shift(lag)
    return data.dropna()

## Evaluation Metrics

We evaluate models using:
- MAPE (relative error)
- RMSE (absolute error)

In [21]:
def evaluate_forecast(y_true, y_pred):
    return {
        "MAPE": mean_absolute_percentage_error(y_true, y_pred),
        "RMSE": root_mean_squared_error(y_true, y_pred)
    }

## Forecasting on Sample SKUs

For demonstration, we apply forecasting to a subset of SKUs
from different segments.

In [22]:
results = []

sample_skus = weekly_df["sku_id"].unique()[:10]

for sku in sample_skus:
    sku_data = weekly_df[weekly_df["sku_id"] == sku].sort_values("date")
    series = sku_data["units_sold"].reset_index(drop=True)

    if len(series) < 20:
        continue

    train, test = train_test_split_time(series)

    # Naive
    naive_pred = naive_forecast(train, test)
    naive_metrics = evaluate_forecast(test, naive_pred)

    # Regression
    lagged = prepare_lag_features(series)
    train_lag, test_lag = train_test_split_time(lagged)

    X_train = train_lag.drop("y", axis=1)
    y_train = train_lag["y"]
    X_test = test_lag.drop("y", axis=1)
    y_test = test_lag["y"]

    model = LinearRegression()
    model.fit(X_train, y_train)
    reg_pred = model.predict(X_test)

    reg_metrics = evaluate_forecast(y_test, reg_pred)

    results.append({
        "sku_id": sku,
        "SKU_segment": sku_data["SKU_segment"].iloc[0],
        "Naive_MAPE": naive_metrics["MAPE"],
        "Reg_MAPE": reg_metrics["MAPE"],
        "Naive_RMSE": naive_metrics["RMSE"],
        "Reg_RMSE": reg_metrics["RMSE"]
    })

## Model Comparison Results

In [23]:
results_df = pd.DataFrame(results)
results_df

,sku_id,SKU_segment,Naive_MAPE,Reg_MAPE,Naive_RMSE,Reg_RMSE
0,SKU0001,AX,0.285309,0.344767,53.195181,60.592364
1,SKU0002,BX,0.348052,0.408041,58.016455,64.427606
2,SKU0003,CX,0.345679,0.426800,48.398911,57.584943
3,SKU0004,BX,0.306554,0.347166,50.656598,54.109424
4,SKU0005,AX,0.276186,0.343154,57.159267,65.417854
5,SKU0006,AX,0.293239,0.337783,48.770334,52.814282
6,SKU0007,AX,0.316499,0.368164,57.024716,62.879793
7,SKU0008,CX,0.300794,0.376357,47.653103,54.848714
8,SKU0009,CX,0.219353,0.284027,36.349315,43.007149
9,SKU0010,AX,0.300670,0.317336,52.638562,53.493826


## Key Forecasting Insights

- Complex models are not always better than simple baselines
- Demand volatility limits forecast accuracy more than model choice
- Forecast error must be explicitly accounted for in inventory decisions

These insights directly inform safety stock calculations
in the next notebook.

## Save Forecast Error Summary

Forecast error statistics will be used to compute safety stock.

In [24]:
results_df.to_csv(f"{DATA_PATH_PROCESSED}/forecast_error_summary.csv", index=False)     

print("Forecast error summary saved successfully")        

Forecast error summary saved successfully


Explicit Train/Test Split for Forecasting results

In [25]:
from sklearn.model_selection import train_test_split

# Ensure data is sorted by date
df = df.sort_values("date")

# Time-based split (last 20% for testing)
split_index = int(len(df) * 0.8)

train_df = df.iloc[:split_index]
test_df = df.iloc[split_index:]

In [ ]:
# Define Features & Target
numeric_cols = df.select_dtypes(include=["int64", "float64"]).columns.tolist()

# Remove target column
feature_cols = [col for col in numeric_cols if col != "units_sold"]

X_train = train_df[feature_cols]
y_train = train_df["units_sold"]

X_test = test_df[feature_cols]
y_test = test_df["units_sold"]

In [27]:
# Train Model
from sklearn.ensemble import RandomForestRegressor

model = RandomForestRegressor(random_state=42)
model.fit(X_train, y_train)

,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""friedman_mse"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""friedman_mse"", which usesmean squared error with Friedman's improvement score for potentialsplits, ""absolute_error"" for the mean absolute error, which minimizesthe L1 loss using the median of each terminal node, and ""poisson"" whichuses reduction in Poisson deviance to find splits.Training using ""absolute_error"" is significantly slowerthan when using ""squared_error""... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

In [28]:
# Generate Predictions
y_pred = model.predict(X_test)

In [29]:
# Save Forecast Results
forecast_df = test_df[["sku_id", "date"]].copy()
forecast_df["actual_demand"] = y_test.values
forecast_df["forecast_demand"] = y_pred

forecast_df.to_csv(
    "../data/processed/forecast_results.csv",
    index=False
)

print("Forecast results saved successfully.")

Forecast results saved successfully.
